In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

from neural_reconstruction.core.topology import TopologyBuilder
from neural_reconstruction.core.pathfinding import PathFinder
from neural_reconstruction.algorithms.fragment_linking.utils import compute_vector_angle,is_direction_too_similar
from neural_reconstruction.core.preprocessing import dilate_epidermis_vertically
from neural_reconstruction.core.evaluation import (
    extract_graph_points,
    compute_average_hausdorff_distance,
    compute_point_min_distances,
    compute_cldice
)
import skimage as ski
# from skimage.measure import label
from scipy.spatial import KDTree
from sklearn.cluster import DBSCAN


In [10]:
BASE_PATH = Path('/home/pony/projects/ienf_q/data_0331')
IMAGE_ID = 'S1585-2_b'

In [11]:
image      = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/image.png',       cv2.IMREAD_COLOR_RGB)[:, :, 1]  # 只取綠色通道
mask       = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/mask.png',        cv2.IMREAD_GRAYSCALE)
annotation = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/weka.png',        cv2.IMREAD_GRAYSCALE)
label      = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/label.png',       cv2.IMREAD_GRAYSCALE)

In [12]:
roi_mask = dilate_epidermis_vertically(mask, offset_px=50)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51))
background = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
image = cv2.subtract(image, background)

roi_image = cv2.bitwise_and(image, image, mask=roi_mask)
roi_annotation = cv2.bitwise_and(annotation, annotation, mask=roi_mask)

# apply opening to roi_annotation to remove small noise
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
# roi_annotation = cv2.morphologyEx(roi_annotation, cv2.MORPH_OPEN, kernel)

# apply closing to roi_annotation to fill small holes
roi_annotation = cv2.morphologyEx(roi_annotation, cv2.MORPH_CLOSE, kernel, iterations=3)
roi_annotation[roi_annotation > 0] = 255


gt_builder = TopologyBuilder()
roi_label = cv2.bitwise_and(label, label, mask=roi_mask)
roi_label = cv2.morphologyEx(roi_label, cv2.MORPH_CLOSE, kernel, iterations=3)
gt_graph = gt_builder.build_seed_graph(roi_label)

/home/pony/projects/ienf_q/src/neural_reconstruction/core/topology/topology_builder.py:125: VisibleDeprecationWarning: separator in column name will change to _ in version 0.13; to silence this warning, use `separator='-'` to maintain current behavior and use `separator='_'` to switch to the new default behavior.
  summary = summarize(skel_obj)


In [13]:
clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(16, 16))
roi_image = clahe.apply(roi_image)

roi_image = ski.filters.sato(roi_image, sigmas=range(3, 8), black_ridges=False)
# min max roi image
roi_image = (roi_image - roi_image.min()) / (roi_image.max() - roi_image.min()) * 255
roi_image = roi_image.astype(np.uint8)

In [14]:
fiber_mask = roi_label > 0
bg_mask    = (roi_label == 0) & (roi_mask > 0)


vals_f = roi_image[roi_label > 0].astype(np.float64)
vals_b = roi_image[bg_mask].astype(np.float64)

mu_f, sig_f = vals_f.mean(), vals_f.std()
mu_b, sig_b = vals_b.mean(), vals_b.std()

denom = sig_f ** 2 + sig_b ** 2

fisher_score = float((mu_f - mu_b) ** 2 / denom)
print(f'Fisher Score: {fisher_score:.4f}')  

Fisher Score: 1.5485


In [15]:
h, w = roi_image.shape
fig_size_x = 128
fig_size_y = int(h / w * fig_size_x)

fig , axes = plt.subplots(1, 1, figsize=(fig_size_x, fig_size_y))
axes.imshow(roi_image)
axes.axis('off')
plt.tight_layout()
plt.savefig(f'enhanced_image.png', bbox_inches='tight', pad_inches=0)
plt.close(fig)